# Phase 3 - Member 4 (Robustness Redesign & Prototype Engineer)

This notebook implements vulnerability-driven defense, prototype fine-tuning, re-attack testing, and final deliverables.

In [1]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFilter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms
from tqdm import tqdm

In [2]:
# Paths + config
PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'Phase 3' else Path.cwd().resolve()
PHASE3_DIR = PROJECT_ROOT / 'Phase 3'
OUT_DIR = PHASE3_DIR / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_ROOT = PROJECT_ROOT / 'dataset'
METADATA_PATH = DATASET_ROOT / 'metadata.csv'
M1_RESULTS_PATH = OUT_DIR / 'strategy_results.csv'
M2_RESULTS_PATH = OUT_DIR / 'min_change_summary.csv'
M3_REPORT_PATH = OUT_DIR / 'blindspot_analysis.md'
M3_CORR_PATH = OUT_DIR / 'm3_attack_correlation.csv'
CHECKPOINT_PATH = PROJECT_ROOT / 'Phase1' / 'cifake_resnet18_final.pt'

ROBUST_MODEL_PATH = OUT_DIR / 'robust_model_prototype.pt'
ROBUSTNESS_CSV_PATH = OUT_DIR / 'robustness_comparison.csv'
FINAL_RECO_PATH = OUT_DIR / 'final_recommendation.md'
AFTER_ATTACK_PATH = OUT_DIR / 'm4_strategy_results_robust.csv'
BEFORE_ATTACK_PATH = OUT_DIR / 'm4_strategy_results_original.csv'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0
EPOCHS = 3
MAX_TRAIN_SAMPLES = 4000
MAX_VAL_SAMPLES = 1000

for p in [METADATA_PATH, M1_RESULTS_PATH, M2_RESULTS_PATH, M3_REPORT_PATH, M3_CORR_PATH, CHECKPOINT_PATH]:
    print(p, 'exists=', p.exists())
print('DEVICE=', DEVICE)

/Users/arulkevin/Desktop/psg_hackathon/Feb-2026-Hackathon/dataset/metadata.csv exists= True
/Users/arulkevin/Desktop/psg_hackathon/Feb-2026-Hackathon/Phase 3/outputs/strategy_results.csv exists= True
/Users/arulkevin/Desktop/psg_hackathon/Feb-2026-Hackathon/Phase 3/outputs/min_change_summary.csv exists= True
/Users/arulkevin/Desktop/psg_hackathon/Feb-2026-Hackathon/Phase 3/outputs/blindspot_analysis.md exists= True
/Users/arulkevin/Desktop/psg_hackathon/Feb-2026-Hackathon/Phase 3/outputs/m3_attack_correlation.csv exists= True
/Users/arulkevin/Desktop/psg_hackathon/Feb-2026-Hackathon/Phase1/cifake_resnet18_final.pt exists= True
DEVICE= cpu


In [3]:
# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [4]:
# Load findings and choose defense strategy
m3_corr = pd.read_csv(M3_CORR_PATH)
m3_report_text = M3_REPORT_PATH.read_text(encoding='utf-8')

if m3_corr.empty:
    primary_attack_family = 'Strategy A (Spatial + Frequency)'
else:
    primary_attack_family = m3_corr.sort_values('n_cases', ascending=False).iloc[0]['attack_family']

if 'Strategy A' in str(primary_attack_family):
    defense_strategy = 'strategy_a_robust_aug'
elif 'Strategy B' in str(primary_attack_family):
    defense_strategy = 'strategy_b_noise_aug'
else:
    text_lower = m3_report_text.lower()
    if 'high-frequency' in text_lower or 'texture' in text_lower:
        defense_strategy = 'strategy_a_robust_aug'
    elif 'noise' in text_lower or 'photometric' in text_lower:
        defense_strategy = 'strategy_b_noise_aug'
    else:
        defense_strategy = 'generic_robust_aug'

print('Primary vulnerability family:', primary_attack_family)
print('Selected defense strategy   :', defense_strategy)

Primary vulnerability family: Strategy A (Spatial + Frequency)
Selected defense strategy   : strategy_a_robust_aug


In [5]:
# Dataset + transforms
CIFAR_MEAN = [0.4914, 0.4822, 0.4465]
CIFAR_STD = [0.2023, 0.1994, 0.2010]

class RobustTransform:
    def __init__(self, strategy: str):
        base = [transforms.Resize((IMG_SIZE, IMG_SIZE))]
        if strategy == 'strategy_a_robust_aug':
            aug = [
                transforms.RandomApply([transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 1.5))], p=0.6),
                transforms.RandomGrayscale(p=0.25),
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.02),
            ]
        elif strategy == 'strategy_b_noise_aug':
            aug = [
                transforms.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.15, hue=0.03),
                transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 0.8))], p=0.3),
            ]
        else:
            aug = [
                transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))], p=0.4),
                transforms.RandomGrayscale(p=0.15),
                transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.1, hue=0.02),
            ]
        self.tfm = transforms.Compose(base + aug + [
            transforms.ToTensor(),
            transforms.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD)
        ])
    def __call__(self, img):
        return self.tfm(img)

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD)
])

class CIFAKEMetadataDataset(Dataset):
    def __init__(self, metadata_df: pd.DataFrame, root_dir: Path, transform=None):
        self.df = metadata_df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.root_dir / str(row['filepath'])
        img = Image.open(img_path).convert('RGB')
        y = int(row['label'])
        x = self.transform(img) if self.transform is not None else transforms.ToTensor()(img)
        return x, y

meta = pd.read_csv(METADATA_PATH)
train_df = meta[meta['split'] == 'train'].copy().sample(frac=1.0, random_state=SEED)
val_df = meta[meta['split'] == 'val'].copy().sample(frac=1.0, random_state=SEED)

if MAX_TRAIN_SAMPLES and len(train_df) > MAX_TRAIN_SAMPLES:
    n_each = MAX_TRAIN_SAMPLES // 2
    train_df = pd.concat([
        train_df[train_df['label'] == 0].head(n_each),
        train_df[train_df['label'] == 1].head(n_each)
    ]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

if MAX_VAL_SAMPLES and len(val_df) > MAX_VAL_SAMPLES:
    n_each = MAX_VAL_SAMPLES // 2
    val_df = pd.concat([
        val_df[val_df['label'] == 0].head(n_each),
        val_df[val_df['label'] == 1].head(n_each)
    ]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

train_ds = CIFAKEMetadataDataset(train_df, DATASET_ROOT, transform=RobustTransform(defense_strategy))
val_ds = CIFAKEMetadataDataset(val_df, DATASET_ROOT, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print('train samples:', len(train_ds), 'val samples:', len(val_ds))

train samples: 4000 val samples: 1000


In [6]:
# Model + fine-tune

def build_model():
    model = models.resnet18(weights=None)
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(model.fc.in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, 2)
    )
    return model


def evaluate_loader(model, loader):
    model.eval()
    total_loss = 0.0
    total = 0
    correct = 0
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            loss = criterion(out, y)
            total_loss += float(loss.item()) * x.size(0)
            preds = out.argmax(dim=1)
            total += y.numel()
            correct += int((preds == y).sum().item())
    return total_loss / max(total, 1), correct / max(total, 1)

base_model = build_model().to(DEVICE)
state_dict = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
base_model.load_state_dict(state_dict)
base_model.eval()

robust_model = build_model().to(DEVICE)
robust_model.load_state_dict(state_dict)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(robust_model.parameters(), lr=3e-5, weight_decay=1e-4)

history = []
for epoch in range(1, EPOCHS + 1):
    robust_model.train()
    running_loss = 0.0
    total = 0
    correct = 0

    for x, y in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}'):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        out = robust_model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        running_loss += float(loss.item()) * x.size(0)
        preds = out.argmax(dim=1)
        total += y.numel()
        correct += int((preds == y).sum().item())

    train_loss = running_loss / max(total, 1)
    train_acc = correct / max(total, 1)
    val_loss, val_acc = evaluate_loader(robust_model, val_loader)
    history.append({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc, 'val_loss': val_loss, 'val_acc': val_acc})
    print(f'Epoch {epoch}: train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.4f}')

torch.save(robust_model.state_dict(), ROBUST_MODEL_PATH)
print('Saved:', ROBUST_MODEL_PATH)
pd.DataFrame(history)

Epoch 1/3:   0%|          | 0/125 [00:00<?, ?it/s]

Epoch 1/3:   1%|          | 1/125 [00:03<07:52,  3.81s/it]

Epoch 1/3:   2%|▏         | 2/125 [00:07<07:34,  3.70s/it]

Epoch 1/3:   2%|▏         | 3/125 [00:10<07:04,  3.48s/it]

Epoch 1/3:   3%|▎         | 4/125 [00:13<06:15,  3.10s/it]

...

Epoch 3/3: 100%|██████████| 125/125 [05:05<00:00,  2.25s/it]

Epoch 3/3: 100%|██████████| 125/125 [05:05<00:00,  2.44s/it]

Epoch 3: train_loss=0.2040, train_acc=0.9215, val_loss=0.1842, val_acc=0.9210


Saved: /Users/arulkevin/Desktop/psg_hackathon/Feb-2026-Hackathon/Phase 3/outputs/robust_model_prototype.pt


,epoch,train_loss,train_acc,val_loss,val_acc
0,1,0.503033,0.75000,0.307403,0.870
1,2,0.282523,0.88725,0.225576,0.903
2,3,0.203999,0.92150,0.184196,0.921


In [7]:
# Attack functions from M1

def apply_blur(image_np, radius):
    pil = Image.fromarray(image_np)
    return np.array(pil.filter(ImageFilter.GaussianBlur(radius=float(radius))))


def apply_frequency_mask(image_np, strength):
    img_float = image_np.astype(np.float32) / 255.0
    dft = np.fft.fft2(img_float, axes=(0, 1))
    dft_shift = np.fft.fftshift(dft, axes=(0, 1))

    h, w, _ = img_float.shape
    mask = np.ones((h, w, 3), np.float32)
    center_h, center_w = h // 2, w // 2
    radius = int(min(h, w) * float(strength))
    mask[center_h-radius:center_h+radius, center_w-radius:center_w+radius] *= 0.5

    dft_shift = dft_shift * mask
    f_ishift = np.fft.ifftshift(dft_shift, axes=(0, 1))
    img_back = np.fft.ifft2(f_ishift, axes=(0, 1))
    img_back = np.abs(img_back)
    img_back = np.clip(img_back, 0, 1)
    return (img_back * 255).astype(np.uint8)


def apply_noise(image_np, sigma, rng):
    noise = rng.normal(0, sigma * 255, image_np.shape)
    noisy = image_np + noise
    return np.clip(noisy, 0, 255).astype(np.uint8)


def apply_brightness(image_np, factor):
    bright = image_np.astype(np.float32) * factor
    return np.clip(bright, 0, 255).astype(np.uint8)


def to_model_tensor(img_np):
    pil = Image.fromarray(img_np)
    x = val_transform(pil).unsqueeze(0).to(DEVICE)
    return x


def predict_details(model, img_np):
    x = to_model_tensor(img_np)
    with torch.no_grad():
        out = model(x)
        probs = F.softmax(out, dim=1)[0]
    pred = int(torch.argmax(probs).item())
    conf = float(probs[pred].item())
    p_fake = float(probs[1].item())
    return pred, conf, p_fake

In [8]:
# Re-run Strategy A/B attacks on same sample set as M1
m1_df = pd.read_csv(M1_RESULTS_PATH)
sample_map = m1_df[['sample_id', 'true_label']].drop_duplicates().reset_index(drop=True)

blur_levels = [0.3, 0.5, 0.8, 1.0]
freq_levels = [0.05, 0.1, 0.15]
noise_levels = [0.01, 0.02, 0.03, 0.05]
brightness_levels = [1.02, 1.05, 1.08]


def eval_model_under_attacks(model, out_path):
    rows = []
    model.eval()

    for _, r in tqdm(sample_map.iterrows(), total=len(sample_map), desc='Attacking'):
        sid = str(r['sample_id'])
        label = str(r['true_label'])
        img_path = DATASET_ROOT / 'test' / label / sid
        if not img_path.exists():
            continue

        image = Image.open(img_path).convert('RGB')
        image_np = np.array(image)

        orig_pred, orig_conf, orig_pfake = predict_details(model, image_np)
        rows.append({'sample_id': sid, 'true_label': label, 'strategy': 'original', 'stage': 'original', 'parameter': 0.0, 'confidence': orig_conf, 'p_fake': orig_pfake, 'flipped': False})

        for blur in blur_levels:
            adv = apply_blur(image_np, blur)
            pred, conf, pfake = predict_details(model, adv)
            rows.append({'sample_id': sid, 'true_label': label, 'strategy': 'A_blur', 'stage': f'blur_{blur}', 'parameter': blur, 'confidence': conf, 'p_fake': pfake, 'flipped': pred != orig_pred})

        for freq in freq_levels:
            adv = apply_blur(image_np, 0.5)
            adv = apply_frequency_mask(adv, freq)
            pred, conf, pfake = predict_details(model, adv)
            rows.append({'sample_id': sid, 'true_label': label, 'strategy': 'A_blur_freq', 'stage': f'blur0.5_freq{freq}', 'parameter': freq, 'confidence': conf, 'p_fake': pfake, 'flipped': pred != orig_pred})

        for sigma in noise_levels:
            rng = np.random.default_rng(abs(hash((sid, label, 'B_noise', sigma))) % (2**32))
            adv = apply_noise(image_np, sigma, rng)
            pred, conf, pfake = predict_details(model, adv)
            rows.append({'sample_id': sid, 'true_label': label, 'strategy': 'B_noise', 'stage': f'noise_{sigma}', 'parameter': sigma, 'confidence': conf, 'p_fake': pfake, 'flipped': pred != orig_pred})

        for bright in brightness_levels:
            rng = np.random.default_rng(abs(hash((sid, label, 'B_noise_bright', bright))) % (2**32))
            adv = apply_noise(image_np, 0.02, rng)
            adv = apply_brightness(adv, bright)
            pred, conf, pfake = predict_details(model, adv)
            rows.append({'sample_id': sid, 'true_label': label, 'strategy': 'B_noise_bright', 'stage': f'noise0.02_bright{bright}', 'parameter': bright, 'confidence': conf, 'p_fake': pfake, 'flipped': pred != orig_pred})

    df = pd.DataFrame(rows)
    df.to_csv(out_path, index=False)
    return df

before_df = eval_model_under_attacks(base_model, BEFORE_ATTACK_PATH)
after_df = eval_model_under_attacks(robust_model, AFTER_ATTACK_PATH)

print('Saved before attacks:', BEFORE_ATTACK_PATH)
print('Saved after attacks :', AFTER_ATTACK_PATH)
print(before_df.shape, after_df.shape)

Attacking:   0%|          | 0/100 [00:00<?, ?it/s]

Attacking:   1%|          | 1/100 [00:00<01:06,  1.48it/s]

Attacking:   2%|▏         | 2/100 [00:00<00:42,  2.30it/s]

Attacking:   3%|▎         | 3/100 [00:01<00:40,  2.42it/s]

Attacking:   4%|▍         | 4/100 [00:01<00:39,  2.40it/s]

...

Attacking:  98%|█████████▊| 98/100 [00:26<00:00,  3.85it/s]

Attacking:  99%|█████████▉| 99/100 [00:26<00:00,  3.68it/s]

Attacking: 100%|██████████| 100/100 [00:26<00:00,  3.29it/s]

Attacking: 100%|██████████| 100/100 [00:26<00:00,  3.73it/s]

Saved before attacks: /Users/arulkevin/Desktop/psg_hackathon/Feb-2026-Hackathon/Phase 3/outputs/m4_strategy_results_original.csv
Saved after attacks : /Users/arulkevin/Desktop/psg_hackathon/Feb-2026-Hackathon/Phase 3/outputs/m4_strategy_results_robust.csv
(1500, 8) (1500, 8)


In [9]:
# Build robustness comparison table

def summarize_attacks(df):
    attack_df = df[df['strategy'] != 'original'].copy()
    base = df[df['strategy'] == 'original'][['sample_id', 'true_label', 'confidence', 'p_fake']].rename(columns={'confidence': 'base_conf', 'p_fake': 'base_p_fake'})
    attack_df = attack_df.merge(base, on=['sample_id', 'true_label'], how='left')
    attack_df['conf_drop'] = attack_df['base_conf'] - attack_df['confidence']
    attack_df['p_fake_abs_delta'] = (attack_df['p_fake'] - attack_df['base_p_fake']).abs()

    sample_strat = attack_df.groupby(['sample_id', 'strategy'], as_index=False)['flipped'].max()
    a_mask = sample_strat['strategy'].isin(['A_blur', 'A_blur_freq'])
    b_mask = sample_strat['strategy'].isin(['B_noise', 'B_noise_bright'])

    flip_rate_a = sample_strat[a_mask]['flipped'].mean() if a_mask.any() else np.nan
    flip_rate_b = sample_strat[b_mask]['flipped'].mean() if b_mask.any() else np.nan
    flip_rate_all = sample_strat['flipped'].mean() if len(sample_strat) else np.nan

    mean_drop_a = attack_df[attack_df['strategy'].isin(['A_blur', 'A_blur_freq'])]['conf_drop'].mean()
    mean_drop_b = attack_df[attack_df['strategy'].isin(['B_noise', 'B_noise_bright'])]['conf_drop'].mean()
    mean_pfake_delta = attack_df['p_fake_abs_delta'].mean()

    return {
        'Flip_Rate_Strategy_A': float(flip_rate_a),
        'Flip_Rate_Strategy_B': float(flip_rate_b),
        'Flip_Rate_Overall': float(flip_rate_all),
        'Mean_Confidence_Drop_A': float(mean_drop_a),
        'Mean_Confidence_Drop_B': float(mean_drop_b),
        'MeanAbsDelta_p_fake_All_Attacks': float(mean_pfake_delta),
    }

before_metrics = summarize_attacks(before_df)
after_metrics = summarize_attacks(after_df)

rows = []
for k in before_metrics:
    rows.append({'Metric': k, 'Original_Model_Value': before_metrics[k], 'Improved_Model_Value': after_metrics[k]})

comparison_df = pd.DataFrame(rows)
comparison_df.to_csv(ROBUSTNESS_CSV_PATH, index=False)
print('Saved:', ROBUSTNESS_CSV_PATH)
comparison_df

Saved: /Users/arulkevin/Desktop/psg_hackathon/Feb-2026-Hackathon/Phase 3/outputs/robustness_comparison.csv


,Metric,Original_Model_Value,Improved_Model_Value
0,Flip_Rate_Strategy_A,0.045000,0.155000
1,Flip_Rate_Strategy_B,0.090000,0.075000
2,Flip_Rate_Overall,0.067500,0.115000
3,Mean_Confidence_Drop_A,-0.029532,-0.010307
4,Mean_Confidence_Drop_B,0.029154,-0.009184
5,MeanAbsDelta_p_fake_All_Attacks,0.046534,0.072111


In [10]:
# Final recommendation markdown

def _rel(path_like):
    p = Path(path_like)
    try:
        return p.resolve().relative_to(PROJECT_ROOT).as_posix()
    except Exception:
        return str(p)

report = []
report.append('# Final Recommendation (Member 4)')
report.append('')
report.append('## Inputs Used')
report.append(f'- M1 baseline attacks: `{_rel(M1_RESULTS_PATH)}`')
report.append(f'- M2 minimal-change summary: `{_rel(M2_RESULTS_PATH)}`')
report.append(f'- M3 blindspot report: `{_rel(M3_REPORT_PATH)}`')
report.append(f'- M3 attack correlation: `{_rel(M3_CORR_PATH)}`')
report.append('')
report.append('## Chosen Defense Strategy')
report.append(f'- Primary vulnerability from M3: `{primary_attack_family}`')
report.append(f'- Implemented defense: `{defense_strategy}`')
if defense_strategy == 'strategy_a_robust_aug':
    report.append('- Applied robust transform with Gaussian blur + random grayscale + mild color jitter to reduce texture/high-frequency over-reliance.')
elif defense_strategy == 'strategy_b_noise_aug':
    report.append('- Applied noise/photometric-focused augmentation to reduce sensitivity to noise and brightness shifts.')
else:
    report.append('- Applied mixed robust augmentation as fallback.')
report.append('')
report.append('## Prototype Training')
report.append(f'- Checkpoint loaded: `{_rel(CHECKPOINT_PATH)}`')
report.append(f'- Fine-tune epochs: `{EPOCHS}` (prototype setting)')
report.append(f'- Train samples used: `{len(train_ds)}`')
report.append(f'- Val samples used: `{len(val_ds)}`')
report.append(f'- Robust model saved: `{_rel(ROBUST_MODEL_PATH)}`')
report.append('')
report.append('## Stress-Test Comparison (Same Attack Families)')
for _, r in comparison_df.iterrows():
    m = r['Metric']
    b = r['Original_Model_Value']
    a = r['Improved_Model_Value']
    report.append(f'- {m}: before={b:.6f}, after={a:.6f}, delta={a-b:+.6f}')
report.append('')
report.append('## Deliverables')
report.append(f'- `robust_model_prototype.pt`: `{_rel(ROBUST_MODEL_PATH)}`')
report.append(f'- `robustness_comparison.csv`: `{_rel(ROBUSTNESS_CSV_PATH)}`')
report.append(f'- `final_recommendation.md`: `{_rel(FINAL_RECO_PATH)}`')

FINAL_RECO_PATH.write_text('\n'.join(report), encoding='utf-8')
print('Saved:', FINAL_RECO_PATH)
print('\n'.join(report[:40]))

Saved: /Users/arulkevin/Desktop/psg_hackathon/Feb-2026-Hackathon/Phase 3/outputs/final_recommendation.md
# Final Recommendation (Member 4)

## Inputs Used
- M1 baseline attacks: `Phase 3/outputs/strategy_results.csv`
- M2 minimal-change summary: `Phase 3/outputs/min_change_summary.csv`
- M3 blindspot report: `Phase 3/outputs/blindspot_analysis.md`
- M3 attack correlation: `Phase 3/outputs/m3_attack_correlation.csv`

## Chosen Defense Strategy
- Primary vulnerability from M3: `Strategy A (Spatial + Frequency)`
- Implemented defense: `strategy_a_robust_aug`
- Applied robust transform with Gaussian blur + random grayscale + mild color jitter to reduce texture/high-frequency over-reliance.

## Prototype Training
- Checkpoint loaded: `Phase1/cifake_resnet18_final.pt`
- Fine-tune epochs: `3` (prototype setting)
- Train samples used: `4000`
- Val samples used: `1000`
- Robust model saved: `Phase 3/outputs/robust_model_prototype.pt`

## Stress-Test Comparison (Same Attack Families)
- Flip_Rate